# 02 — Correlation Analysis

**Primary:** Tianyi Qin  
**Support:** Tuan Wei

This notebook uses the exact processed rows, target and split created in `01_preprocessing.ipynb`.

Designed variable set:
- property size: `accommodates`, `bedrooms`, `bathrooms`;
- location: `distance_cbd_km`;
- amenities: `amenity_count`;
- target: `high_price`.

All four required methods are computed for every unique pair: **Pearson, Spearman, Mutual Information (MI), and Normalised Mutual Information (NMI)**.

## 1. Imports and processed data

In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Run notebooks/01_preprocessing.ipynb first to create processed_listings.csv."
    )

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target counts:", df["high_price"].value_counts().sort_index().to_dict())

Processed shape: (14472, 20)
Split counts: {'train': 11577, 'test': 2895}
Target counts: {0: 10854, 1: 3618}


## 2. Variable set and method implementation

Pearson and Spearman are computed directly on numeric representations.

For MI/NMI, continuous/count variables are discretised into up to five quantile bins. The binary target is left as binary. This makes the pairwise MI/NMI calculation symmetric and reproducible; the report should explicitly state this implementation choice.

In [2]:
VARIABLES = [
    "accommodates",
    "bedrooms",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "high_price",
]

missing_cols = [c for c in VARIABLES if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing correlation columns: {missing_cols}")

VARIABLE_TYPES = {
    "accommodates": "numeric",
    "bedrooms": "numeric",
    "bathrooms": "numeric",
    "distance_cbd_km": "numeric",
    "amenity_count": "numeric",
    "high_price": "binary",
}

display(df[VARIABLES].describe())

,accommodates,bedrooms,bathrooms,distance_cbd_km,amenity_count,high_price
count,14472.000000,14102.000000,14471.000000,14472.000000,14472.000000,14472.000000
mean,4.554381,2.070132,1.485315,10.354091,41.024323,0.250000
std,2.515616,1.097852,0.742045,14.335911,14.544795,0.433028
min,1.000000,1.000000,0.000000,0.015709,0.000000,0.000000
25%,2.000000,1.000000,1.000000,1.137856,33.000000,0.000000
50%,4.000000,2.000000,1.000000,4.019169,43.000000,0.000000
75%,6.000000,3.000000,2.000000,13.306550,51.000000,0.250000
max,16.000000,12.000000,12.000000,79.999033,91.000000,1.000000


## 3. Helpers for MI/NMI discretisation

In [3]:
def to_information_categories(series, variable_type, max_bins=5):
    if variable_type == "binary":
        return pd.to_numeric(series, errors="coerce")

    numeric = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    valid = numeric.notna()

    if valid.sum() < 2:
        return out

    # qcut may drop duplicate edges for low-cardinality numeric variables.
    binned = pd.qcut(
        numeric.loc[valid],
        q=min(max_bins, numeric.loc[valid].nunique()),
        labels=False,
        duplicates="drop",
    )
    out.loc[valid] = binned.astype(float)
    return out

## 4. Compute all four methods for every pair

In [4]:
rows = []

for a, b in combinations(VARIABLES, 2):
    pair = df[[a, b]].dropna().copy()

    pearson = pearsonr(pair[a], pair[b]).statistic
    spearman = spearmanr(pair[a], pair[b]).statistic

    a_disc = to_information_categories(
        pair[a], VARIABLE_TYPES[a]
    )
    b_disc = to_information_categories(
        pair[b], VARIABLE_TYPES[b]
    )
    valid = a_disc.notna() & b_disc.notna()

    mi = mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )
    nmi = normalized_mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )

    rows.append({
        "var_a": a,
        "var_b": b,
        "n": len(pair),
        "pearson": float(pearson),
        "spearman": float(spearman),
        "mutual_information": float(mi),
        "normalised_mutual_information": float(nmi),
        "implementation_note": (
            "Pearson/Spearman on numeric values; MI/NMI on 5-quantile "
            "discretisation for numeric variables; high_price left binary."
        ),
    })

corr_results = pd.DataFrame(rows)
display(corr_results)

,var_a,var_b,n,pearson,spearman,mutual_information,normalised_mutual_information,implementation_note
0,accommodates,bedrooms,14102,0.877774,0.883191,0.403374,0.385163,Pearson/Spearman on numeric values; MI/NMI on ...
1,accommodates,bathrooms,14471,0.667793,0.650323,0.257380,0.217648,Pearson/Spearman on numeric values; MI/NMI on ...
2,accommodates,distance_cbd_km,14472,0.231651,0.148605,0.052714,0.035766,Pearson/Spearman on numeric values; MI/NMI on ...
3,accommodates,amenity_count,14472,0.222542,0.236885,0.031952,0.021693,Pearson/Spearman on numeric values; MI/NMI on ...
4,accommodates,high_price,14472,0.472557,0.451785,0.106751,0.112332,Pearson/Spearman on numeric values; MI/NMI on ...
5,bedrooms,bathrooms,14101,0.730220,0.696254,0.235556,0.262694,Pearson/Spearman on numeric values; MI/NMI on ...
6,bedrooms,distance_cbd_km,14102,0.295063,0.245952,0.086805,0.073437,Pearson/Spearman on numeric values; MI/NMI on ...
7,bedrooms,amenity_count,14102,0.153830,0.179863,0.012064,0.010219,Pearson/Spearman on numeric values; MI/NMI on ...
8,bedrooms,high_price,14102,0.517891,0.496979,0.126359,0.191129,Pearson/Spearman on numeric values; MI/NMI on ...
9,bathrooms,distance_cbd_km,14471,0.175997,0.184188,0.052292,0.039672,Pearson/Spearman on numeric values; MI/NMI on ...


## 5. Target associations

This table isolates every predictor–target pair so the group can identify the strongest/weakest associations with actual values.

In [5]:
target_rows = corr_results[
    (corr_results["var_a"] == "high_price")
    | (corr_results["var_b"] == "high_price")
].copy()

target_rows["predictor"] = np.where(
    target_rows["var_a"] == "high_price",
    target_rows["var_b"],
    target_rows["var_a"],
)
target_rows["abs_pearson"] = target_rows["pearson"].abs()
target_rows["abs_spearman"] = target_rows["spearman"].abs()

display(
    target_rows[
        [
            "predictor",
            "n",
            "pearson",
            "spearman",
            "mutual_information",
            "normalised_mutual_information",
        ]
    ].sort_values("normalised_mutual_information", ascending=False)
)

,predictor,n,pearson,spearman,mutual_information,normalised_mutual_information
8,bedrooms,14102,0.517891,0.496979,0.126359,0.191129
11,bathrooms,14471,0.454505,0.452333,0.106053,0.133476
4,accommodates,14472,0.472557,0.451785,0.106751,0.112332
13,distance_cbd_km,14472,0.151772,0.158402,0.013411,0.012350
14,amenity_count,14472,0.093304,0.106652,0.010542,0.009716


## 6. Predictor–predictor relationships

This is used to identify potential redundancy/multicollinearity before modelling.

In [6]:
predictor_pairs = corr_results[
    (corr_results["var_a"] != "high_price")
    & (corr_results["var_b"] != "high_price")
].copy()

predictor_pairs["abs_pearson"] = predictor_pairs["pearson"].abs()
predictor_pairs["abs_spearman"] = predictor_pairs["spearman"].abs()

method_applicability = pd.DataFrame([
    {
        "method": "Pearson",
        "applicable": True,
        "interpretation_limit": (
            "All analysed columns are numeric; with binary high_price this is "
            "the point-biserial correlation. Sensitive to non-linearity and outliers."
        ),
    },
    {
        "method": "Spearman",
        "applicable": True,
        "interpretation_limit": (
            "Applicable to monotonic numeric relationships and less sensitive to outliers; "
            "ties are common in count and binary variables."
        ),
    },
    {
        "method": "Mutual Information",
        "applicable": True,
        "interpretation_limit": (
            "Applied after five-quantile discretisation of numeric variables; "
            "magnitude depends on binning and is not on the correlation scale."
        ),
    },
    {
        "method": "Normalised Mutual Information",
        "applicable": True,
        "interpretation_limit": (
            "Applied to the same discretised categories; useful for relative comparison "
            "but still sensitive to binning and ties."
        ),
    },
])

high_correlation_decisions = predictor_pairs.loc[
    (predictor_pairs["abs_pearson"] >= 0.80)
    | (predictor_pairs["abs_spearman"] >= 0.80),
    ["var_a", "var_b", "pearson", "spearman", "n"],
].copy()
high_correlation_decisions["analysis_role"] = (
    "Post-hoc held-out sensitivity analysis only; not used for model selection."
)
high_correlation_decisions["downstream_decision"] = (
    "Retain both in the prespecified size benchmark, then quantify held-out "
    "macro-F1 sensitivity after dropping each member of the pair."
)

display(
    predictor_pairs.sort_values(
        ["abs_pearson", "normalised_mutual_information"],
        ascending=False,
    )
)
distance_association_rows = []
for predictor, representation in [
    ("latitude", "before: raw coordinate"),
    ("longitude", "before: raw coordinate"),
    ("distance_cbd_km", "after: derived CBD distance"),
]:
    pair = df[[predictor, "high_price"]].dropna()
    distance_association_rows.append({
        "predictor": predictor,
        "representation": representation,
        "n": len(pair),
        "pearson_with_high_price": float(pearsonr(pair[predictor], pair["high_price"]).statistic),
        "spearman_with_high_price": float(spearmanr(pair[predictor], pair["high_price"]).statistic),
    })

distance_target_association = pd.DataFrame(distance_association_rows)
distance_target_association["analysis_scope"] = (
    "All eligible rows; exploratory association only, not model selection."
)

display(method_applicability)
display(distance_target_association)
display(high_correlation_decisions)


,var_a,var_b,n,pearson,spearman,mutual_information,normalised_mutual_information,implementation_note,abs_pearson,abs_spearman
0,accommodates,bedrooms,14102,0.877774,0.883191,0.403374,0.385163,Pearson/Spearman on numeric values; MI/NMI on ...,0.877774,0.883191
5,bedrooms,bathrooms,14101,0.730220,0.696254,0.235556,0.262694,Pearson/Spearman on numeric values; MI/NMI on ...,0.730220,0.696254
1,accommodates,bathrooms,14471,0.667793,0.650323,0.257380,0.217648,Pearson/Spearman on numeric values; MI/NMI on ...,0.667793,0.650323
6,bedrooms,distance_cbd_km,14102,0.295063,0.245952,0.086805,0.073437,Pearson/Spearman on numeric values; MI/NMI on ...,0.295063,0.245952
2,accommodates,distance_cbd_km,14472,0.231651,0.148605,0.052714,0.035766,Pearson/Spearman on numeric values; MI/NMI on ...,0.231651,0.148605
3,accommodates,amenity_count,14472,0.222542,0.236885,0.031952,0.021693,Pearson/Spearman on numeric values; MI/NMI on ...,0.222542,0.236885
9,bathrooms,distance_cbd_km,14471,0.175997,0.184188,0.052292,0.039672,Pearson/Spearman on numeric values; MI/NMI on ...,0.175997,0.184188
7,bedrooms,amenity_count,14102,0.153830,0.179863,0.012064,0.010219,Pearson/Spearman on numeric values; MI/NMI on ...,0.153830,0.179863
10,bathrooms,amenity_count,14471,0.128707,0.153770,0.014592,0.011078,Pearson/Spearman on numeric values; MI/NMI on ...,0.128707,0.153770
12,distance_cbd_km,amenity_count,14472,0.028157,0.005708,0.006373,0.003962,Pearson/Spearman on numeric values; MI/NMI on ...,0.028157,0.005708


,method,applicable,interpretation_limit
0,Pearson,True,All analysed columns are numeric; with binary ...
1,Spearman,True,Applicable to monotonic numeric relationships ...
2,Mutual Information,True,Applied after five-quantile discretisation of ...
3,Normalised Mutual Information,True,Applied to the same discretised categories; us...


,predictor,representation,n,pearson_with_high_price,spearman_with_high_price,analysis_scope
0,latitude,before: raw coordinate,14472,-0.023395,-0.044620,All eligible rows; exploratory association onl...
1,longitude,before: raw coordinate,14472,0.101116,0.087377,All eligible rows; exploratory association onl...
2,distance_cbd_km,after: derived CBD distance,14472,0.151772,0.158402,All eligible rows; exploratory association onl...


,var_a,var_b,pearson,spearman,n,analysis_role,downstream_decision
0,accommodates,bedrooms,0.877774,0.883191,14102,Post-hoc held-out sensitivity analysis only; n...,Retain both in the prespecified size benchmark...


## 7. Correlation matrices and figures

In [7]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

corr_results.to_csv(TABLE_OUT / "correlation_results.csv", index=False)
target_rows.to_csv(TABLE_OUT / "correlation_target_associations.csv", index=False)
predictor_pairs.to_csv(TABLE_OUT / "correlation_predictor_pairs.csv", index=False)
method_applicability.to_csv(TABLE_OUT / "correlation_method_applicability.csv", index=False)
distance_target_association.to_csv(TABLE_OUT / "preprocessing_distance_target_association.csv", index=False)
high_correlation_decisions.to_csv(TABLE_OUT / "correlation_high_pair_decisions.csv", index=False)

def symmetric_matrix(results, value_col):
    matrix = pd.DataFrame(
        np.eye(len(VARIABLES)),
        index=VARIABLES,
        columns=VARIABLES,
        dtype=float,
    )

    # MI has a meaningful non-one diagonal, but the diagonal is not used
    # in pairwise interpretation. Keep 1.0 visually for consistency.
    for _, row in results.iterrows():
        matrix.loc[row["var_a"], row["var_b"]] = row[value_col]
        matrix.loc[row["var_b"], row["var_a"]] = row[value_col]
    return matrix

for method, col in {
    "pearson": "pearson",
    "spearman": "spearman",
    "mi": "mutual_information",
    "nmi": "normalised_mutual_information",
}.items():
    matrix = symmetric_matrix(corr_results, col)
    matrix.to_csv(TABLE_OUT / f"{method}_matrix.csv")

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix.values, aspect="auto")
    ax.set_xticks(range(len(VARIABLES)))
    ax.set_yticks(range(len(VARIABLES)))
    ax.set_xticklabels(VARIABLES, rotation=45, ha="right")
    ax.set_yticklabels(VARIABLES)
    ax.set_title(f"{method.upper()} pairwise matrix")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_OUT / f"correlation_{method}.png", dpi=200)
    plt.close(fig)

print("Saved correlation tables to:", TABLE_OUT.relative_to(REPO_ROOT))
print("Saved correlation figures to:", FIG_OUT.relative_to(REPO_ROOT))

Saved correlation tables to: output/tables
Saved correlation figures to: output/figures


## 8. Evidence checklist for group-written interpretation

Use the generated tables to write the report yourselves. The reproducible evidence records:
- why each method is applicable and its dataset-specific limitation;
- that the correlation analysis uses all eligible rows, including the held-out split, as a descriptive/exploratory analysis only;
- raw latitude/longitude versus derived CBD-distance associations with `high_price`;
- the strongest, weakest and near-absent predictor–target relationships;
- the strong `accommodates`–`bedrooms` relationship and a separately labelled post-hoc held-out drop-one sensitivity check;
- plausible confounders/biases to discuss using association rather than causal language.

Neither the all-row correlation table nor the held-out sensitivity check is used to select model hyperparameters; those are selected from training-only cross-validation.
